In [ ]:
# Install required libraries

import sys

!{sys.executable} -m pip install -U langchain langchain-community langchain-google-genai langchain-huggingface faiss-cpu pypdf

In [2]:
# Import libraries and configure Gemini API

import os

from getpass import getpass

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("Setup completed successfully.")

C:\Users\gogos\AppData\Local\Temp\ipykernel_7388\437815549.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Enter your Gemini API key:  ········


Setup completed successfully.


In [3]:
# Load PDF documents

loader = PyPDFDirectoryLoader(".")

documents = loader.load()

print(f"Loaded {len(documents)} documents.")

Loaded 162 documents.


In [4]:
# Clean extracted text

import re

for doc in documents:
    doc.page_content = re.sub(r"[ \t]+", " ", doc.page_content)
    doc.page_content = re.sub(r"\n\s*\n+", "\n\n", doc.page_content)
    doc.page_content = "\n".join(
        line.strip() for line in doc.page_content.splitlines()
    )

In [5]:
# Split documents into overlapping chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 911


In [6]:
# Generate embeddings for document chunks

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Number of vectors:", len(vectors))
print("Vector dimension:", len(vectors[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of vectors: 911
Vector dimension: 384


In [7]:
    vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector store created successfully.")

Vector store created successfully.


In [8]:
# Create a retriever

retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

In [9]:
# Retrieve relevant chunks for a query

query = "How can I control Varroa mites?"

retrieved_chunks = retriever.invoke(query)

print("Retrieved chunks:", len(retrieved_chunks))

Retrieved chunks: 5


In [10]:
# Generate an answer using Gemini

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using the provided context whenever possible.

If the context contains enough information to answer the question,
answer based on the context and start your response with:
[SUPPORTED]

If the context does not contain enough information to answer the question,
you may answer using your general knowledge. In that case, start your
response with:
[NOT VERIFIED]

When using general knowledge, clearly state that the information is not
verified against the knowledge base.

Do not present information from your general knowledge as if it came from
the provided documents.

Context:
{context}

Question:
{question}
""")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [ ]:
# Connect retrieval, prompt, and LLM

context = "\n\n".join(
    chunk.page_content for chunk in retrieved_chunks
)

messages = prompt.invoke({
    "context": context,
    "question": query
})

response = llm.invoke(messages)

answer = response.content[0]["text"]

In [12]:
# Display answer, source status, and retrieved documents

if answer.startswith("[SUPPORTED]"):
    source_status = "Supported by knowledge base"
    answer = answer.replace("[SUPPORTED]", "", 1).strip()

elif answer.startswith("[NOT VERIFIED]"):
    source_status = "Not verified against knowledge base"
    answer = answer.replace("[NOT VERIFIED]", "", 1).strip()

else:
    source_status = "Source status not determined"

print("Answer:")
print(answer)

print("\nSource status:")
print(source_status)

print("\nRetrieved documents:")

sources = sorted(
    set(chunk.metadata.get("source") for chunk in retrieved_chunks)
)

for source in sources:
    print(f"- {source}")

Answer:
Based on the provided context, Varroa mites can be managed and controlled using the following strategies:

* **Use a combination of control options:** Employ a mix of organic and synthetic chemicals, cultural and mechanical practices, and genetic practices. Do not rely on a single chemical or management practice.
* **Alternate treatments:** Alternate between different chemical controls and varying active ingredients to help slow the development of chemical resistance in mites. 
* **Treatment options include:**
  1. Synthetic miticides
  2. Organic miticides
  3. Cultural and mechanical practices
  4. Genetic practices
* **Timing:** When treating an apiary, treat all colonies in the apiary at the same time once one colony reaches the treatment threshold. 
* **Monitoring:** Use alcohol/soap washes and powdered sugar shakes as accurate monitoring methods.

Source status:
Supported by knowledge base

Retrieved documents:
- HBHC-Guide_Varroa-Mgmt_8thEd-081622.pdf
- HBHC_Hive_BMPs_v1